In [11]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 273 nodes, deleted 253 relationships, completed after 45 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 655 ms.


### Node Rules 

In [102]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env.json")
# env = Environment("../dtgraph/type_checking/env_common_movies.json")


generate_humans = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(:Movie)
GENERATE
(x = (p):Actor {
    name = p.name,
    born = p.born,
    old = convert_age("123")
})
''',
env=env,
type_strict = True)

# common_movies = Rule('''
# MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)
# WHERE id(x) < id(z)
# GENERATE
# ((x):Actor {
#     name = x.name
# })-[(x,z):ACTED_WITH {
#     CommonMovies = [y.title]
# }]->((z):Actor {
#     name = z.name
# })
# ''',
# env=env,
# type_strict = True
# )


# generate_created = Rule('''
# MATCH (p:Person)-[:DIRECTED]->(m:Movie)
# GENERATE
# (x = (p):)-[():CREATED {
#     role = 123
# }]->(y = (m):)
# ''',
# env=env,
# type_strict = True)

### Execute Rules

In [103]:
my_transform = Transformation([generate_humans])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 17 ms.
AST:
PropertyAccess
    ├── var: p
    └── prop: name
AST:
PropertyAccess
    ├── var: p
    └── prop: born
AST:
FunctionCall: convert_age
    └── Literal
        ├── value: 123
        └── type: string


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Unknown function 'convert_age' (line 9, column 17 (offset: 188))
"        x.old = convert_age("123")"
                 ^} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

### Abort Transformation

In [90]:
my_transform.abort()

Index: Removed 1 index, completed after 7 ms.
Abort: Deleted 102 nodes, deleted 0 relationships, completed after 55 ms.
